In [31]:
# !pip install -q -U \
#   torch \
#   torchvision \
#   torchaudio \
#   transformers \
#   datasets \
#   accelerate \
#   peft \
#   trl \
#   bitsandbytes \
#   sentencepiece \
#   safetensors \
#   evaluate \
#   scikit-learn

In [32]:
# pip install cuda-toolkit==12.6.0

In [33]:
# pip install torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121

In [34]:
from pathlib import Path
import os
import sys
import json
import time
import torch
import pandas as pd

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

In [35]:
if os.path.exists("/content"):
    PROJECT_ROOT = Path("/content")
else:
    PROJECT_ROOT = Path(".")

DATA_DIR = PROJECT_ROOT / "data"
SFT_DIR = DATA_DIR / "sft_ready"
SPLIT_DIR = DATA_DIR / "splits"

RUN_DIR = PROJECT_ROOT / "outputs" / "lora_runs" / "lora_llama3_feina30_v1"
ADAPTER_DIR = RUN_DIR / "final_adapter"
EVAL_DIR = RUN_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

TEST_JSONL = SFT_DIR / "feina_30_test_sft.jsonl"
TEST_ORIG_CSV = SPLIT_DIR / "feina_clean_30_test.csv"

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

print("TEST_JSONL :", TEST_JSONL)
print("TEST_ORIG_CSV:", TEST_ORIG_CSV)
print("ADAPTER_DIR:", ADAPTER_DIR)
print("EVAL_DIR   :", EVAL_DIR)

TEST_JSONL : /content/data/sft_ready/feina_30_test_sft.jsonl
TEST_ORIG_CSV: /content/data/splits/feina_clean_30_test.csv
ADAPTER_DIR: /content/outputs/lora_runs/lora_llama3_feina30_v1/final_adapter
EVAL_DIR   : /content/outputs/lora_runs/lora_llama3_feina30_v1/evaluation


In [36]:
data_files = {
    "test": str(TEST_JSONL),
}

dataset = load_dataset("json", data_files=data_files)
test_ds = dataset["test"]

print("test_ds:", test_ds)
print(test_ds[0].keys())
print(test_ds[0]["instruction"][:500])

test_ds: Dataset({
    features: ['row_id', 'instruction', 'output', 'text'],
    num_rows: 238
})
dict_keys(['row_id', 'instruction', 'output', 'text'])
Reescribe en español el siguiente texto con lenguaje más claro y sencillo.
Conserva el significado original y no inventes información.

Devuelve solo la versión final simplificada.

Texto:
Todo lo anterior está ligado también al desorden con que muchas personas y familias desarrollan sus vidas, careciendo de toda sistematicidad para estar controlando su patrimonio, por pequeño que sea, así como sus ingresos y sus egresos.

Versión simplificada:


In [37]:
def recover_source_text_from_instruction(instruction: str) -> str:
    text = str(instruction)

    marker = "Texto:"
    end_marker = "Versión simplificada:"

    if marker in text:
        text = text.split(marker, 1)[1]

    if end_marker in text:
        text = text.split(end_marker, 1)[0]

    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = " ".join(text.split()).strip()

    return text

In [38]:
compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

bnb_config

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}

In [39]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("pad_token:", tokenizer.pad_token)
print("eos_token:", tokenizer.eos_token)

pad_token: <|eot_id|>
eos_token: <|eot_id|>


In [40]:
t0 = time.perf_counter()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto",
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)

model.eval()

t1 = time.perf_counter()
print(f"Tiempo cargando modelo + adapter: {t1 - t0:.2f} s")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Tiempo cargando modelo + adapter: 95.48 s


In [41]:
def clean_lora_output(decoded_text: str) -> str:
    text = str(decoded_text).strip()

    marker = "Versión simplificada:"
    if marker in text:
        text = text.split(marker, 1)[-1].strip()

    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = " ".join(text.split()).strip()

    return text

In [42]:
def generate_simplification(instruction: str, max_new_tokens: int = 256) -> str:
    inputs = tokenizer(
        instruction,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    cleaned = clean_lora_output(decoded)
    return cleaned

In [43]:
for i in range(3):
    row = test_ds[i]
    pred = generate_simplification(row["instruction"], max_new_tokens=256)

    print("=" * 90)
    print("ROW ID:", row["row_id"])
    print("\nINSTRUCTION:\n", row["instruction"][:500])
    print("\nREFERENCE:\n", row["output"][:500])
    print("\nPREDICTION:\n", pred[:500])

Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ROW ID: 75

INSTRUCTION:
 Reescribe en español el siguiente texto con lenguaje más claro y sencillo.
Conserva el significado original y no inventes información.

Devuelve solo la versión final simplificada.

Texto:
Todo lo anterior está ligado también al desorden con que muchas personas y familias desarrollan sus vidas, careciendo de toda sistematicidad para estar controlando su patrimonio, por pequeño que sea, así como sus ingresos y sus egresos.

Versión simplificada:

REFERENCE:
 Lo anterior se relaciona con que muchas personas y familias desarrollan sus vidas desordenadamente. Es decir, carecen de sistematicidad para estar controlando su patrimonio, sus ingresos y sus egresos.

PREDICTION:
 Todo lo anterior está ligado al desorden con que muchas personas y familias desarrollan sus vidas. Carecen de sistematicidad para controlar su patrimonio, por pequeño que sea, así como sus ingresos y sus egresos.


Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ROW ID: 93

INSTRUCTION:
 Reescribe en español el siguiente texto con lenguaje más claro y sencillo.
Conserva el significado original y no inventes información.

Devuelve solo la versión final simplificada.

Texto:
Este comportamiento puede obedecer a varias causas, mencionamos dos que consideramos sustanciales: la primera, que somos producto de una educación, que como ya lo hemos expresado, no nos educa en planificación ni en finanzas para la vida, considerando tales conocimientos y habilidades exclusivos para especiali

REFERENCE:
 Este comportamiento tiene varias causas, aunque mencionamos dos esenciales. En primer lugar, somos producto de una educación que no nos educa en planificación ni en finanzas para la vida. Esto porque se considera como conocimientos y habilidades exclusivos para especialistas. En segundo lugar, muchas personas sienten miedo al descubrir cuánto y cómo gastan cada mes, principalmente por pertenecer a una sociedad cada vez más consumista. Respecto a lo anterio

In [44]:
records = []

t0 = time.perf_counter()

for i, row in enumerate(test_ds):
    pred = generate_simplification(row["instruction"], max_new_tokens=256)

    records.append({
        "row_id": row["row_id"],
        "instruction": row["instruction"],
        "generated_text": pred,
    })

    if (i + 1) % 25 == 0:
        print(f"Procesados: {i + 1}/{len(test_ds)}")

t1 = time.perf_counter()
print(f"Tiempo total de generación en test: {(t1 - t0)/60:.2f} min")

lora_test_df = pd.DataFrame(records)
display(lora_test_df.head(3))

Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Procesados: 25/238


Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Procesados: 50/238


Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

...

Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Procesados: 225/238


Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Tiempo total de generación en test: 20.91 min


,row_id,instruction,generated_text
0,75,Reescribe en español el siguiente texto con le...,Todo lo anterior está ligado al desorden con q...
1,93,Reescribe en español el siguiente texto con le...,Este comportamiento puede obedecer a varias ca...
2,94,Reescribe en español el siguiente texto con le...,"Ante sus temores, esas personas prefieren vivi..."


In [45]:
lora_test_df["source_text"] = lora_test_df["instruction"].apply(recover_source_text_from_instruction)

test_df_from_sft = pd.DataFrame(test_ds)
test_df_from_sft = test_df_from_sft.rename(columns={"output": "reference_text"})

lora_test_df = lora_test_df.merge(
    test_df_from_sft[["row_id", "reference_text"]],
    on="row_id",
    how="left"
)

print(lora_test_df.shape)
display(lora_test_df[["row_id", "source_text", "reference_text", "generated_text"]].head(3))

(238, 5)


,row_id,source_text,reference_text,generated_text
0,75,Todo lo anterior está ligado también al desord...,Lo anterior se relaciona con que muchas person...,Todo lo anterior está ligado al desorden con q...
1,93,Este comportamiento puede obedecer a varias ca...,"Este comportamiento tiene varias causas, aunqu...",Este comportamiento puede obedecer a varias ca...
2,94,"Dichas personas, ante sus temores, prefieren v...",Dichas personas prefieren vivir en la oscurida...,"Ante sus temores, esas personas prefieren vivi..."


In [46]:
pred_path = EVAL_DIR / "lora_test_predictions.csv"
lora_test_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

print("Predicciones guardadas en:", pred_path)

Predicciones guardadas en: /content/outputs/lora_runs/lora_llama3_feina30_v1/evaluation/lora_test_predictions.csv


In [48]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.evaluation.metrics import evaluate_dataframe, summarize_metrics

In [49]:
evaluated_lora_test_df = evaluate_dataframe(
    lora_test_df,
    source_col="source_text",
    pred_col="generated_text",
    ref_col="reference_text",
)

display(evaluated_lora_test_df.head(3))

,row_id,instruction,generated_text,source_text,reference_text,src_words,pred_words,ref_words,src_sentences,pred_sentences,...,additions_proportion,deletions_proportion,inflesz_pred,inflesz_src,inflesz_delta,rouge1_f,rouge2_f,rougeL_f,bertscore_f1,sbert_similarity
0,75,Reescribe en español el siguiente texto con le...,Todo lo anterior está ligado al desorden con q...,Todo lo anterior está ligado también al desord...,Lo anterior se relaciona con que muchas person...,37,34,29,1,2,...,0.058824,0.135135,57.905588,35.132297,22.773291,None,None,None,None,None
1,93,Reescribe en español el siguiente texto con le...,Este comportamiento puede obedecer a varias ca...,Este comportamiento puede obedecer a varias ca...,"Este comportamiento tiene varias causas, aunqu...",95,76,82,1,5,...,0.026316,0.221053,62.116579,-28.503947,90.620526,None,None,None,None,None
2,94,Reescribe en español el siguiente texto con le...,"Ante sus temores, esas personas prefieren vivi...","Dichas personas, ante sus temores, prefieren v...",Dichas personas prefieren vivir en la oscurida...,41,41,38,1,1,...,0.024390,0.024390,39.715488,39.715488,0.000000,None,None,None,None,None


In [50]:
lora_summary = summarize_metrics(
    evaluated_lora_test_df,
    group_cols=[],
)

display(lora_summary)

,sari,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,sentence_splits,levenshtein_similarity,exact_copy,additions_proportion,deletions_proportion,inflesz_pred,inflesz_src,inflesz_delta
0,38.085834,72.750557,76.029628,-3.279071,0.876656,0.285714,0.824622,0.210084,0.050156,0.16494,56.407447,49.079043,7.328404


In [51]:
print("Columnas de lora_summary:")
print(lora_summary.columns.tolist())

Columnas de lora_summary:
['sari', 'fernandez_huerta_pred', 'fernandez_huerta_src', 'fernandez_huerta_delta', 'compression_ratio_eval', 'sentence_splits', 'levenshtein_similarity', 'exact_copy', 'additions_proportion', 'deletions_proportion', 'inflesz_pred', 'inflesz_src', 'inflesz_delta']


In [52]:
main_metric_cols = [
    col for col in [
        "sari",
        "bertscore_f1",
        "bleu",
        "rouge1",
        "rouge2",
        "rougeL",
        "fernandez_huerta",
        "inflesz",
        "compression_ratio_eval",
        "exact_copy",
    ]
    if col in lora_summary.columns
]

lora_main_metrics = lora_summary[main_metric_cols].copy()
display(lora_main_metrics)

,sari,compression_ratio_eval,exact_copy
0,38.085834,0.876656,0.210084


In [53]:
evaluated_path = EVAL_DIR / "lora_test_evaluated.csv"
summary_path = EVAL_DIR / "lora_test_summary.csv"

evaluated_lora_test_df.to_csv(evaluated_path, index=False, encoding="utf-8-sig")
lora_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Evaluated:", evaluated_path)
print("Summary  :", summary_path)

Evaluated: /content/outputs/lora_runs/lora_llama3_feina30_v1/evaluation/lora_test_evaluated.csv
Summary  : /content/outputs/lora_runs/lora_llama3_feina30_v1/evaluation/lora_test_summary.csv


In [54]:
lora_summary = summarize_metrics(
    evaluated_lora_test_df,
    group_cols=[],
)

display(lora_summary)

,sari,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,sentence_splits,levenshtein_similarity,exact_copy,additions_proportion,deletions_proportion,inflesz_pred,inflesz_src,inflesz_delta
0,38.085834,72.750557,76.029628,-3.279071,0.876656,0.285714,0.824622,0.210084,0.050156,0.16494,56.407447,49.079043,7.328404


In [55]:
print(lora_summary.columns.tolist())

['sari', 'fernandez_huerta_pred', 'fernandez_huerta_src', 'fernandez_huerta_delta', 'compression_ratio_eval', 'sentence_splits', 'levenshtein_similarity', 'exact_copy', 'additions_proportion', 'deletions_proportion', 'inflesz_pred', 'inflesz_src', 'inflesz_delta']


In [56]:
main_metric_cols = [
    col for col in [
        "sari",
        "bertscore_f1",
        "bleu",
        "rouge1",
        "rouge2",
        "rougeL",
        "fernandez_huerta",
        "inflesz",
        "compression_ratio_eval",
        "exact_copy",
    ]
    if col in lora_summary.columns
]

lora_main_metrics = lora_summary[main_metric_cols].copy()
display(lora_main_metrics)

,sari,compression_ratio_eval,exact_copy
0,38.085834,0.876656,0.210084


In [57]:
summary_path = EVAL_DIR / "lora_test_summary.csv"
evaluated_path = EVAL_DIR / "lora_test_evaluated.csv"

lora_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
evaluated_lora_test_df.to_csv(evaluated_path, index=False, encoding="utf-8-sig")

print("Summary guardado en:", summary_path)
print("Evaluated guardado en:", evaluated_path)

Summary guardado en: /content/outputs/lora_runs/lora_llama3_feina30_v1/evaluation/lora_test_summary.csv
Evaluated guardado en: /content/outputs/lora_runs/lora_llama3_feina30_v1/evaluation/lora_test_evaluated.csv


### Resumen del Notebook 13

* **Carga del modelo:** Se cargó el modelo base junto con el adapter **LoRA** entrenado.
* **Generación:** Se procesaron las simplificaciones sobre el conjunto de **test**.
* **Limpieza de datos:** Se filtraron las salidas para conservar únicamente la simplificación generada.
* **Integración:** Se unieron las predicciones con el `source_text` y `reference_text` del split original.
* **Evaluación:** Se aplicó el mismo pipeline de métricas establecido en el proyecto.
* **Objetivo:** Comparar el rendimiento del **LoRA** frente al mejor baseline basado en prompts (*prompt-based*).